# 05 — Fairness Audit

**Layer:** Governance · **Day:** 4 · **Authoritative script:** `scripts/day4_governance.py`

## Objective

Document the model's **group-level behaviour** so a CHRO or risk committee can decide whether and how to deploy it. This is a governance checkpoint — not a deployment approval.

## Inputs

- `lakehouse/silver/employees.parquet` — demographics and `AttritionFlag`
- `lakehouse/gold/fact_attrition_risk.parquet` — model `RiskScore` and `RiskBand`

## Outputs

- `docs/fairness_audit_summary.csv` — high-risk prediction rate and disparate-impact ratio per group
- `docs/fairness_auc_by_group.csv` — per-group ROC-AUC where both classes are present
- `docs/images/fairness_audit.png` — disparate-impact bar chart with the 0.8 / 1.25 reference lines
- `docs/model_card.md` — refreshed with the latest metrics and review notes

## Method

1. Compute the **overall** high-risk rate (the model's positive prediction rate).
2. For each protected/diagnostic group (`Gender`, `AgeBand`, `Department`):
   - Compute the per-group high-risk rate.
   - Compute the **disparate-impact ratio** = group rate / overall rate.
3. Where both attrition classes are present in a group, compute per-group ROC-AUC.
4. Render a bar chart with reference lines at the conventional 0.8 and 1.25 thresholds.

## What "good" looks like

All groups inside the 0.8 – 1.25 band, with stable per-group AUC. Any group outside that band is a flag for human review — not an automatic rejection.

In [ ]:
from pathlib import Path
import pandas as pd

DOCS = Path('../docs')
summary = DOCS / 'fairness_audit_summary.csv'
auc = DOCS / 'fairness_auc_by_group.csv'
print('summary exists:', summary.exists())
print('auc exists:', auc.exists())

In [ ]:
if summary.exists():
    df = pd.read_csv(summary)
    df['disparate_impact_ratio'] = df['disparate_impact_ratio'].round(3)
    df['high_risk_rate'] = df['high_risk_rate'].round(3)
    df.sort_values('disparate_impact_ratio')

In [ ]:
if auc.exists():
    pd.read_csv(auc).round(3)

## Interpretation

- **The audit is diagnostic, not normative.** It cannot tell you whether the model is *fair*; it can tell you where to look.
- The 0.8 / 1.25 thresholds come from EEOC "four-fifths" practice — pragmatic, not statutory.
- Per-group AUC degradation is often more revealing than disparate-impact ratios on small samples.

In a real deployment, any flagged group triggers:

1. A data-quality and labelling check on that cohort.
2. A feature-importance review (with SHAP) for that cohort.
3. A decision: retrain, re-threshold, restrict use, or do not deploy.

## Interview talking points

- This notebook is intentionally short — it is the **artefact**, not the analysis. The analysis lives in conversation with HR, legal, and risk.
- The CSVs and PNG are the outputs that go into the model-review pack; nothing here is hidden behind interactive widgets.
- The `docs/model_card.md` is regenerated by the same script (`scripts/day4_governance.py`) so the audit, the metrics, and the model description never drift.